# Wairagkar 2025 · voice synthesis T15

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `t15_neural_data.zip` from [Dryad](https://doi.org/10.5061/dryad.2280gb64f). Choose the May 8, 2026 release. Create `studies/20250612_wairagkar/data/raw/dryad_2280gb64f/` and save the download as `studies/20250612_wairagkar/data/raw/dryad_2280gb64f/dryad_2280gb64f__t15_neural_data.zip`.

### 2. Extract for inspection (optional)

The release is one ZIP and the pipeline reads it directly: the export, inference, and evaluation commands open `studies/20250612_wairagkar/data/raw/dryad_2280gb64f/dryad_2280gb64f__t15_neural_data.zip` themselves. Extract a copy only to inspect the files by hand, into `studies/20250612_wairagkar/data/extracted/dryad_2280gb64f/`.

```text
python -m zipfile -e "studies/20250612_wairagkar/data/raw/dryad_2280gb64f/dryad_2280gb64f__t15_neural_data.zip" "studies/20250612_wairagkar/data/extracted/dryad_2280gb64f"
```

After extraction, check for `studies/20250612_wairagkar/data/extracted/dryad_2280gb64f/Dryad_T15_all/t15_day025/`. Each day folder holds `<day>_neural_data.mat`, `<day>_speech_indices.mat`, and `<day>_target_speech_feats.mat`. Evaluation days also hold the decoder that ran that day, such as `t15_day195/brain2voice_t15_day193.h5`.

### 3. Prepare the model input

The inference and training scripts read neural data and acoustic targets directly from `studies/20250612_wairagkar/data/raw/dryad_2280gb64f/dryad_2280gb64f__t15_neural_data.zip`. They prepare arrays in memory. The ZIP is the pipeline's input.

Set up the upstream encoder and vocoder code before running the pipeline.

```text
python shared/tools/setup_upstream.py --study 20250612_wairagkar
```

### 4. Export the common session file

Write one recording day to the session format shared by every study in this repository. The file stores each trial with the neural features, the acoustic targets, and the word and syllable timings as released. This example exports day 195, one of the two days in the archived evaluation.

```text
python studies/20250612_wairagkar/code/export_sessions.py --session t15_day195
```

**Output** `studies/20250612_wairagkar/data/processed/t15/dryad_2280gb64f_day195.h5`. Add `--list` to see every day and whether it is exported, or use `--all` to export them all.

Trace every trial back to the ZIP with one command, then check that the neural features and the acoustic targets of each trial share one time axis.

```text
python shared/tools/validate_session.py studies/20250612_wairagkar/data/processed/t15/dryad_2280gb64f_day195.h5 --source trials
python studies/20250612_wairagkar/code/check_session_timing.py studies/20250612_wairagkar/data/processed/t15/dryad_2280gb64f_day195.h5
```

**Output** The first command ends with `source trace: 305/305 trials re-read and identical`. The second prints `neural_frames_match: 305`, `acoustic_frames_match: 305`, and `intervals_inside_trial: 305`.


In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "shared"))
from session_format import read_session

session_file = root / "studies/20250612_wairagkar/data/processed/t15/dryad_2280gb64f_day195.h5"
if session_file.is_file():
    with read_session(session_file) as session:
        print(f"{session.attrs['session']}: {len(session)} trials, "
              f"decoder {session.attrs.get('brain2voice_decoder')}")
        trial = session.where(lambda t: t.attrs["trial_description"] == "Evaluation trial")[0]
        print(trial.trial_id, "·", trial.sentence)
        for name, array in {**trial.neural(), **trial.target()}.items():
            if hasattr(array, "shape"):
                print(f"  {name}: {array.shape} {array.dtype}")
        print("  word onset/offset frames:", trial.timing("word").tolist())
else:
    print("Session file not found. Run the export command above to create it.")


t15_day195: 305 trials, decoder brain2voice_t15_day193.h5
t15_day195/data/0146 · I am where I need to be right now.
  spike_pow: (1198, 256) float32
  tx: (1198, 256) float32
  lpcnet_syllable: (1198, 20) float32
  lpcnet_word: (1198, 20) float32
  word onset/offset frames: [[121, 239], [256, 336], [392, 475], [509, 567], [618, 656], [726, 768], [820, 872], [925, 985], [1022, 1106]]


## Read the archived results

The acoustic output is continuous speech. The reproduction archives per-day mean Mel-band correlations. The paper's listener word-transcription median is a separate Figure 2c value.

This walkthrough uses the checked-in archive and takes no data download.

In [2]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [3]:
voice = root / "studies/20250612_wairagkar/results/20261124_import_voice_synthesis/tier2_mel_full_trial.json"
results = json.loads(voice.read_text(encoding="utf-8"))
for day, metrics in results["days"].items():
    r = metrics["mel_band_r_mean"]
    print(f"{day}: {metrics['n_scored']} scored trials, r = {r:.3f}, 100 × (1 − r) = {100 * (1-r):.1f}%")
paper = next(row for row in benchmark["panel_cd_online_wer_speed"] if row["label"].startswith("Wairagkar"))
print(f"Paper's separate listener median: {paper['wer_mean']}% WER")

t15_day179: 20 scored trials, r = 0.612, 100 × (1 − r) = 38.8%
t15_day195: 20 scored trials, r = 0.686, 100 × (1 − r) = 31.4%
Paper's separate listener median: 43.75% WER


## Continue the full reproduction

`t15_control_experiment.zip` is a separate release file.

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`export_sessions.py`](code/export_sessions.py), [`infer.py`](code/infer.py), [`evaluate.py`](code/evaluate.py). Run a command with `--help` to see its arguments.